# Canadian Unemployment Analysis and Forecasting (2005–2025)

**Goal:** analyze unemployment trends in Canada, build predictive models, and give data-driven policy recommendations.

**Data:** Statistics Canada, *Labour force characteristics by province, monthly, seasonally adjusted* — Table 14-10-0287-01
([download](https://www150.statcan.gc.ca/n1/tbl/csv/14100287-eng.zip)). Analysis period: January 2005 to December 2025.

**Author:** Krish Sabapathy — Data Science & AI Specialist program, final project.

### Corrections made in this version
1. **Filtering by `Statistics` and `Data type`.** The table stores several rows per month for every series: the estimate itself plus its standard errors, and seasonally adjusted plus unadjusted versions. The earlier version averaged all of them together, which pulled the "unemployment rate" down to about 2–6%. This version keeps only **Estimate** and **Seasonally adjusted** rows, so the rate matches Statistics Canada's published figures.
2. **Age-group analysis** now uses the unemployment rate only (the earlier chart mixed in population counts) and compares non-overlapping groups: 15–24, 25–54 and 55+.
3. **Lag and rolling features** are built on one national monthly series in date order, and the rolling mean uses only past months, so no future information leaks into the model.
4. **Forecast dates** step by month, not by day.
5. **One threshold** (7.0%) is used and justified consistently for the high/low unemployment target.
6. **The second classifier** is a Decision Tree throughout, and feature importance comes from that same model.

In [ ]:
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, r2_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

## Section 1 — Data Understanding

### Load the data
The full Statistics Canada file is large, so it is downloaded once and then read in chunks, keeping only the rows this analysis needs.

In [ ]:
DATA_URL = "https://www150.statcan.gc.ca/n1/tbl/csv/14100287-eng.zip"
ZIP_PATH = "14100287-eng.zip"
CSV_PATH = "14100287.csv"

if not os.path.exists(CSV_PATH):
    if not os.path.exists(ZIP_PATH):
        print("Downloading from Statistics Canada ...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extract(CSV_PATH)
print("Data file ready:", CSV_PATH)

### Q1 — Dataset breakdown: list all columns

In [ ]:
preview = pd.read_csv(CSV_PATH, nrows=5, low_memory=False)
print(preview.columns.tolist())
preview

| Column | Meaning | Example |
|---|---|---|
| REF_DATE | Month the value refers to | 2005-01 |
| GEO | Geography (Canada or a province) | Ontario |
| DGUID | Unique ID for the geographic area | 2021A000011124 |
| Labour force characteristics | Which measure the row holds | Unemployment rate |
| Gender | Gender group | Total - Gender, Men+, Women+ |
| Age group | Age group | 15 to 24 years |
| Statistics | Estimate, or a standard error of the estimate | Estimate |
| Data type | Seasonally adjusted, unadjusted or trend-cycle | Seasonally adjusted |
| UOM / UOM_ID | Unit of measure and its code | Percentage / 239 |
| SCALAR_FACTOR / SCALAR_ID | Whether values are scaled (e.g. thousands) | units |
| VECTOR | Unique ID of one data series | v2062815 |
| COORDINATE | Position of the series in the table | 1.7.1.1.1.1 |
| VALUE | The number itself | 7.1 |
| STATUS / SYMBOL | Data-quality flags | E (use with caution) |
| TERMINATED | Whether the series has ended | t |
| DECIMALS | Decimal places in VALUE | 1 |

**Key point:** each month has several rows per series — the estimate, its standard errors, and both seasonally adjusted and unadjusted versions. They must be filtered before any averaging.

### Q2 — Data cleaning

In [ ]:
KEEP_MEASURES = ["Unemployment rate", "Unemployment"]
WANTED = {"REF_DATE", "GEO", "Labour force characteristics", "Gender", "Sex",
          "Age group", "Statistics", "Data type", "VALUE"}

chunks = []
for chunk in pd.read_csv(CSV_PATH, usecols=lambda c: c in WANTED,
                         chunksize=500_000, low_memory=False):
    chunk = chunk.rename(columns={"Sex": "Gender"})   # older versions of the table use "Sex"
    chunk = chunk[
        chunk["Labour force characteristics"].isin(KEEP_MEASURES)
        & (chunk["Statistics"] == "Estimate")
        & (chunk["Data type"] == "Seasonally adjusted")
    ]
    chunks.append(chunk)
raw = pd.concat(chunks, ignore_index=True)
print("Rows kept:", len(raw))

In [ ]:
# Clean column names
df = raw.rename(columns={
    "REF_DATE": "date",
    "GEO": "region",
    "Gender": "gender",
    "Age group": "age_group",
    "Labour force characteristics": "category",
    "VALUE": "value",
}).drop(columns=["Statistics", "Data type"])

# Convert REF_DATE to datetime and keep 2005-2025
df["date"] = pd.to_datetime(df["date"], format="%Y-%m")
df = df[(df["date"] >= "2005-01-01") & (df["date"] <= "2025-12-31")]

# Handle missing values: convert to numbers, drop blanks and duplicates
df["value"] = pd.to_numeric(df["value"], errors="coerce")
before = len(df)
df = df.dropna(subset=["value"]).drop_duplicates()
print(f"Dropped {before - len(df)} rows with missing or duplicate values")

df = df.sort_values("date").reset_index(drop=True)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
df.head()

In [ ]:
def get_series(region="Canada", gender="Total - Gender", age="15 years and over",
               category="Unemployment rate"):
    """Return one monthly series as a date-indexed Series."""
    s = df[(df["region"] == region) & (df["gender"] == gender)
           & (df["age_group"] == age) & (df["category"] == category)]
    return s.set_index("date")["value"].sort_index()

canada = get_series()
print("Canada, 15+, months:", len(canada))
print(canada.describe().round(2))

**Why is proper time formatting critical?** The whole analysis depends on month-to-month order. Dates stored as text sort incorrectly, which breaks trends, lag features, rolling averages and the time-based train/test split. Labour data is also seasonal, and those patterns only show on a correct timeline.

### Q3 — Concepts
- **Employment vs labour force vs unemployment:** the labour force is everyone working or actively looking for work. Employment and unemployment are its two parts: labour force = employed + unemployed.
- **Participation rate vs employment rate:** participation rate is the share of the working-age population in the labour force; employment rate is the share that actually has a job.
- **Unemployment rate vs actual numbers:** the rate is unemployed ÷ labour force (a percentage, comparable across regions); the count is the number of people. The count can rise while the rate stays flat if the labour force grows.

## Section 2 — Exploratory Data Analysis

### Q4 — Time trend

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(canada.index, canada.values, label="Monthly rate")
ax.plot(canada.groupby(canada.index.year).transform("mean"), linestyle="--",
        label="Yearly average")
ax.set_title("Canada unemployment rate, 15 years and over (seasonally adjusted)")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

yearly = canada.groupby(canada.index.year).mean().round(2)
print("Yearly average unemployment rate (%):")
print(yearly.to_string())

In [ ]:
# Identify peaks and drops
peak_date = canada.idxmax()
print(f"Highest rate: {canada.max():.1f}% in {peak_date:%B %Y}")
print(f"Lowest rate:  {canada.min():.1f}% in {canada.idxmin():%B %Y}")

gfc = canada["2008-01":"2010-12"]
print(f"Financial-crisis peak: {gfc.max():.1f}% in {gfc.idxmax():%B %Y}")

count = get_series(category="Unemployment")   # persons, in thousands
print(f"Most unemployed people: {count.max():,.0f} thousand in {count.idxmax():%B %Y}")

**What real-world events explain these changes?**
- **2008–2009:** the global financial crisis reduced hiring and caused layoffs, pushing unemployment up.
- **2010–2019:** a gradual recovery and a long, fairly stable period.
- **2020:** COVID-19 closures caused the sharpest spike in the series, followed by a fast recovery in 2021–2022 as restrictions lifted.
- **2023 onward:** unemployment has been rising again as interest rates stayed high, hiring slowed and the labour force grew quickly.

### Q5 — Regional analysis

In [ ]:
prov = df[(df["region"] != "Canada") & (df["category"] == "Unemployment rate")
          & (df["gender"] == "Total - Gender") & (df["age_group"] == "15 years and over")]

prov_stats = (prov.groupby("region")["value"]
                  .agg(mean_rate="mean", std_dev="std")
                  .round(2)
                  .sort_values("mean_rate", ascending=False))
print(prov_stats)

prov_stats["mean_rate"].sort_values().plot(kind="barh", figsize=(9, 5),
                                           title="Average unemployment rate by province, 2005–2025")
plt.xlabel("Unemployment rate (%)")
plt.show()

print("Highest unemployment:", prov_stats["mean_rate"].idxmax())
print("Most stable (lowest standard deviation):", prov_stats["std_dev"].idxmin())

**Why:** provinces that depend on natural resources and seasonal work (fishing, oil, tourism), such as Newfoundland and Labrador, tend to have the highest and most volatile unemployment. Provinces with more diversified, year-round economies tend to be lower and steadier.

### Q6 — Age-group analysis

In [ ]:
AGE_GROUPS = ["15 to 24 years", "25 to 54 years", "55 years and over"]

fig, ax = plt.subplots(figsize=(12, 5))
for age in AGE_GROUPS:
    s = get_series(age=age)
    ax.plot(s.index, s.values, label=age)
ax.set_title("Unemployment rate by age group, Canada")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

age_means = pd.Series({age: get_series(age=age).mean() for age in AGE_GROUPS}).round(2)
print("Average rate by age group (%):")
print(age_means.to_string())
print("Most affected:", age_means.idxmax(), "| Least affected:", age_means.idxmin())

**Interpretation:** young workers (15–24) are the most affected. They hold more part-time, entry-level and temporary jobs, which are cut first in downturns. Workers aged 25–54 and 55+ are more often in stable, long-term positions.

### Q7 — Gender analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
gender_means = {}
for g in ["Men+", "Women+"]:
    s = get_series(gender=g)
    gender_means[g] = round(s.mean(), 2)
    ax.plot(s.index, s.values, label=g)
ax.set_title("Unemployment rate by gender, Canada, 15 years and over")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("Average rate by gender (%):", gender_means)

**Are there consistent differences?** Men's unemployment is usually somewhat higher and swings more in recessions, because men are concentrated in cyclical industries such as construction, manufacturing and resources. Women are more represented in education and health care, which are steadier. COVID-19 was an exception: retail, hospitality and caregiving were hit hard.

## Section 3 — Regression

### Q8 — Feature engineering
The model uses the national monthly series. New features: **year**, **month**, **lag_1–lag_3** (the rate 1–3 months earlier) and **rolling_mean_3** (average of the previous 3 months, shifted so it uses past data only).

In [ ]:
model_df = canada.rename("value").to_frame()
model_df["year"] = model_df.index.year
model_df["month"] = model_df.index.month
for k in (1, 2, 3):
    model_df[f"lag_{k}"] = model_df["value"].shift(k)
model_df["rolling_mean_3"] = model_df["value"].shift(1).rolling(3).mean()
model_df = model_df.dropna()

FEATURES = ["year", "month", "lag_1", "lag_2", "lag_3", "rolling_mean_3"]
TARGET = "value"
model_df.head()

**Why are lag features powerful?** Unemployment moves gradually, so last month's value is the best single clue to this month's. Lags let the model capture momentum and persistence in the series.

### Q9 and Q10 — Train models with a time-based split
**Why not a random split?** Shuffling mixes future months into training, so the model would "learn the future" and look better than it really is. A time-based split trains on earlier months and tests on later months, the same way a real forecast works.

In [ ]:
split = int(len(model_df) * 0.8)
train, test = model_df.iloc[:split], model_df.iloc[split:]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Train: {train.index.min():%Y-%m} to {train.index.max():%Y-%m} ({len(train)} months)")
print(f"Test:  {test.index.min():%Y-%m} to {test.index.max():%Y-%m} ({len(test)} months)")

lin_reg = LinearRegression().fit(X_train, y_train)
tree_reg = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)
pred_lin = lin_reg.predict(X_test)
pred_tree = tree_reg.predict(X_test)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, y_test, label="Actual", linewidth=2)
ax.plot(test.index, pred_lin, label="Linear regression", linestyle="--")
ax.plot(test.index, pred_tree, label="Decision tree", linestyle=":")
ax.set_title("Test period: actual vs predicted unemployment rate")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Q11 — Evaluation

In [ ]:
results = pd.DataFrame({
    "R2": [r2_score(y_test, pred_lin), r2_score(y_test, pred_tree)],
    "MAE": [mean_absolute_error(y_test, pred_lin), mean_absolute_error(y_test, pred_tree)],
}, index=["Linear regression", "Decision tree"]).round(3)
print(results)

results.plot(kind="bar", subplots=True, layout=(1, 2), figsize=(10, 4), legend=False, rot=0)
plt.tight_layout()
plt.show()

best = results["MAE"].idxmin()
print(f"Best model (lowest MAE): {best}")

**Which model is best and why?** Higher R² and lower MAE are better. A decision tree cannot predict values outside the range it saw in training and gives step-shaped predictions, while linear regression extrapolates smoothly from the lag features. On a slowly moving series like unemployment, linear regression is usually the stronger model.

### Q12 — Forecast the next 12 months

In [ ]:
HORIZON = 12
final_model = LinearRegression().fit(model_df[FEATURES], model_df[TARGET])

history = canada.copy()
future = []
for _ in range(HORIZON):
    next_date = history.index[-1] + pd.DateOffset(months=1)
    row = pd.DataFrame([{
        "year": next_date.year,
        "month": next_date.month,
        "lag_1": history.iloc[-1],
        "lag_2": history.iloc[-2],
        "lag_3": history.iloc[-3],
        "rolling_mean_3": history.iloc[-3:].mean(),
    }])
    pred = final_model.predict(row[FEATURES])[0]
    future.append((next_date, pred))
    history.loc[next_date] = pred

forecast = pd.Series(dict(future)).round(2)
print(forecast.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
recent = canada.iloc[-60:]
ax.plot(recent.index, recent.values, label="Historical")
ax.plot([recent.index[-1], *forecast.index], [recent.iloc[-1], *forecast.values],
        linestyle="--", label="Forecast")
ax.set_title("12-month unemployment rate forecast, Canada")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

**How reliable are these predictions?** They are estimates, not exact values. Each forecast month feeds the next, so error builds up over the horizon, and the model cannot see shocks such as policy changes or recessions. The first few months are the most trustworthy; treat the later months as a direction of travel.

## Section 4 — Classification

### Q13 — Create the target
**High unemployment = rate ≥ 7.0%.** Over 2005–2025, Canada's rate typically sits between about 5.5% and 8%, and its long-run median is close to 7%. Months at or above 7% therefore mark a labour market that is weaker than usual, such as the years after the 2008–09 financial crisis and the pandemic period. The code below prints the median and the share of high months so the choice can be checked.

In [ ]:
THRESHOLD = 7.0
model_df["high_unemployment"] = (model_df["value"] >= THRESHOLD).astype(int)

print(f"Median rate 2005-2025: {canada.median():.2f}%")
print(f"Share of months at or above {THRESHOLD}%: {model_df['high_unemployment'].mean():.1%}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.step(model_df.index, model_df["high_unemployment"], where="post")
ax.set_yticks([0, 1], ["Low", "High"])
ax.set_title(f"High-unemployment months (rate >= {THRESHOLD}%)")
plt.show()

### Q14 — Train models: logistic regression and decision tree

In [ ]:
y_cls = model_df["high_unemployment"]
X_train_c, X_test_c = model_df[FEATURES].iloc[:split], model_df[FEATURES].iloc[split:]
y_train_c, y_test_c = y_cls.iloc[:split], y_cls.iloc[split:]
print("High months in test set:", int(y_test_c.sum()), "of", len(y_test_c))
if y_test_c.sum() == 0:
    print("Note: the test period has no high-unemployment months at this threshold, "
          "so accuracy alone will look perfect. Report recall on the training period too.")

log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_model.fit(X_train_c, y_train_c)
pred_log = log_model.predict(X_test_c)

tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clf.fit(X_train_c, y_train_c)
pred_tree_c = tree_clf.predict(X_test_c)

### Q15 — Evaluation

In [ ]:
for name, pred in [("Logistic regression", pred_log), ("Decision tree", pred_tree_c)]:
    tn, fp, fn, tp = confusion_matrix(y_test_c, pred, labels=[0, 1]).ravel()
    print(f"{name}: accuracy {accuracy_score(y_test_c, pred):.1%} | "
          f"TN={tn} FP={fp} FN={fn} TP={tp}")
    print(classification_report(y_test_c, pred, labels=[0, 1],
                                target_names=["Low", "High"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, [("Logistic regression", pred_log),
                                    ("Decision tree", pred_tree_c)]):
    ConfusionMatrixDisplay.from_predictions(y_test_c, pred, labels=[0, 1],
                                            display_labels=["Low", "High"],
                                            ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

**What errors are being made?** False positives (a normal month flagged as high) and false negatives (a high month missed).

**Which error is more serious?** False negatives. Missing a real rise in unemployment delays policy responses; a false alarm only causes extra caution.

### Q16 — Probabilities with `predict_proba()`

In [ ]:
proba_log = log_model.predict_proba(X_test_c)[:, 1]
proba_tree = tree_clf.predict_proba(X_test_c)[:, 1]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(X_test_c.index, proba_log, label="Logistic regression")
ax.plot(X_test_c.index, proba_tree, label="Decision tree")
ax.axhline(0.5, linestyle="--", color="grey")
ax.set_title("Predicted probability of high unemployment, test period")
ax.set_ylabel("Probability")
ax.legend()
plt.show()

`predict_proba()` returns the model's estimated probability for each class instead of just the label. For example, [0.72, 0.28] means a 72% chance of "low" and 28% of "high".

**In policy terms,** a probability expresses risk. A rising probability of high unemployment can trigger early preparation, such as readying training or income-support programs, before the rate actually crosses the threshold.

## Section 5 — Advanced Analysis

### Q17 — Correlation

In [ ]:
corr = model_df[FEATURES + ["value"]].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

print(corr["value"].drop("value").sort_values(ascending=False).round(2))

**What influences unemployment?** The lag features and the rolling mean correlate most strongly with the current rate: unemployment is driven largely by its own recent history. Month and year correlate weakly, because they capture seasonality and long-term drift rather than short-term movement.

### Q18 — Feature importance (decision tree classifier)

In [ ]:
importance = pd.Series(tree_clf.feature_importances_, index=FEATURES).sort_values()
print(importance.round(3).sort_values(ascending=False))

importance.plot(kind="barh", figsize=(8, 4), title="Feature importance, decision tree classifier")
plt.show()

plt.figure(figsize=(18, 8))
plot_tree(tree_clf, feature_names=FEATURES, class_names=["Low", "High"],
          filled=True, rounded=True, fontsize=8)
plt.show()

**Impact:** the tree splits mainly on the most recent months (lag and rolling-mean features): if unemployment was already high last month, it is very likely still high. Month and year add little, reflecting seasonal effects and long economic cycles.

## Section 6 — Policy Insights

### Q19 — Key findings

In [ ]:
print("Summary of results")
print(f"- Canada's unemployment rate peaked at {canada.max():.1f}% in {canada.idxmax():%B %Y}.")
print(f"- Highest-unemployment province: {prov_stats['mean_rate'].idxmax()} "
      f"({prov_stats['mean_rate'].max():.1f}% average).")
print(f"- Most stable province: {prov_stats['std_dev'].idxmin()}.")
print(f"- Most affected age group: {age_means.idxmax()} ({age_means.max():.1f}% average).")
print(f"- Best regression model: {best} (MAE {results.loc[best, 'MAE']:.2f} points).")
print(f"- Logistic regression accuracy: {accuracy_score(y_test_c, pred_log):.1%}; "
      f"decision tree: {accuracy_score(y_test_c, pred_tree_c):.1%}.")

- **Trends:** unemployment rises in clusters during shocks (2008–09, 2020) and falls gradually in recoveries; it has been rising again since 2023.
- **High-risk periods:** when the rate has already been climbing for several months, the chance of a high-unemployment period rises sharply.
- **Vulnerable groups:** young workers, people in seasonal or temporary jobs, and residents of resource-dependent provinces.

### Q20 — Recommendations
- **Youth employment:** expand paid internships, apprenticeships and co-op placements; strengthen career guidance; fund short, job-focused training in digital skills and the trades.
- **Regional interventions:** invest in infrastructure and local industry in high-unemployment provinces, and help resource-dependent regions diversify.
- **Labour-force participation:** affordable childcare, flexible work and retraining for older workers, faster recognition of newcomers' foreign credentials, and inclusive hiring.

### Q21 — Risk analysis: what if unemployment rises sharply?
- **Economy:** lower household spending and tax revenue, slower growth.
- **Government:** higher spending on Employment Insurance and social support while revenue falls.
- **Society:** financial stress, mental-health impacts and widening regional and age-group inequality.

Early-warning signals from the classification model can help governments act before these effects take hold.